In [ ]:
# -*- coding: utf-8 -*-
"""
TESTE — RF ORIGINAL AGINDO EM 25–60 kHz vs PARK, COM ANÁLISE EM JANELAS PEQUENAS
================================================================================

Versão simplificada do seu código:

    - mantém apenas RF_original_wide;
    - mantém Park como comparação;
    - remove todas as outras variantes do RF;
    - RF e Park compensam em 25–60 kHz;
    - métricas e classificação são calculadas nas janelas pequenas perto dos picos;
    - validação Leave-One-Temperature-Out.

Saída:
    resultados_RF_original_wide25_60_vs_Park_janelas_picos_RAPIDO
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(42)


# ============================================================
# 2) CONFIGURAÇÕES GERAIS
# ============================================================

ARQ_BASE = "base-completo--.pkl"

PASTA_SAIDA = "resultados_RF_original_wide25_60_vs_Park_janelas_picos_RAPIDO"
PASTA_CSV = os.path.join(PASTA_SAIDA, "csvs")
PASTA_GRAFICOS = os.path.join(PASTA_SAIDA, "graficos")
PASTA_CONFUSAO = os.path.join(PASTA_GRAFICOS, "matrizes_confusao")
PASTA_CURVAS = os.path.join(PASTA_GRAFICOS, "curvas_exemplo")
PASTA_METRICAS = os.path.join(PASTA_GRAFICOS, "metricas_por_temperatura")

for p in [PASTA_SAIDA, PASTA_CSV, PASTA_GRAFICOS, PASTA_CONFUSAO, PASTA_CURVAS, PASTA_METRICAS]:
    os.makedirs(p, exist_ok=True)

REF_TEMP = 30.0

# RF/Park agem nesta faixa larga.
COMP_FREQ_MIN_KHZ = 25.0
COMP_FREQ_MAX_KHZ = 60.0

# RF prevê a correção numa malha reduzida e interpola de volta.
USAR_RF_DOWNSAMPLE_SAIDA = True
RF_MAX_OUTPUT_POINTS = 4500
RF_DOWNSAMPLE_STEP_MIN = 1

# Park busca o shift numa malha reduzida, mas aplica na curva completa.
USAR_PARK_DOWNSAMPLE_FIT = True
PARK_FIT_MAX_POINTS = 4500

PRINT_PROGRESS_METHODS = True

# Busca das janelas.
FREQ_PEAK_SEARCH_MIN_KHZ = 25.0
FREQ_PEAK_SEARCH_MAX_KHZ = 60.0
PEAK_WINDOW_WIDTH_KHZ = 3.0
TOP_N_PEAK_WINDOWS = 10
MIN_DIST_ENTRE_PICOS_KHZ = 3.0
MIN_FREQ_POINTS_WINDOW = 8
PEAK_CENTERS_MANUAL_KHZ = []

# Mantém sua J2 manual.
INCLUIR_JANELA_J2_MANUAL = True
JANELA_J2_MANUAL = (38.7, 41.7)

# Validação.
STRICT_LOTO_COMPENSATION = True
USAR_REFERENCIA_GLOBAL = True

# Classificação.
FEATURE_SETS = ["metricas_RMSD_CCDM", "curva_e_metricas"]
FEATURE_SET_PREFERIDO = "metricas_RMSD_CCDM"

TEMP_EXEMPLO_CURVAS = 80
SALVAR_PDF = True

DANOS = [0, 1, 2]
LABELS_MULTI = [0, 1, 2]
LABELS_MULTI_TXT = ["D0", "D1", "D2"]
LABELS_BIN = [0, 1]
LABELS_BIN_TXT = ["Sem dano", "Com dano"]


# ============================================================
# 3) SOMENTE RF ORIGINAL + PARK
# ============================================================

SMOOTH_WIN_PADRAO = 5

RF_VARIANTS = {
    "RF_original_wide": {
        "nome": "RF original 25–60",
        "smooth_win": 5,
        "params": dict(
            n_estimators=250,
            max_depth=10,
            min_samples_leaf=2,
            min_samples_split=4,
            max_features="sqrt",
            n_jobs=-1,
            random_state=0,
        ),
    },
}

# Park também fica no teste.
PARK_MAX_SHIFT_FRAC = 0.18
PARK_SMOOTH_WIN = 5
PARK_NSTEPS = 151

METODOS_RF = ["RF_original_wide"]
METODOS_TODOS = ["RF_original_wide", "Park"]

NOME_METODO = {
    "RF_original_wide": "RF original 25–60",
    "Park": "Park 25–60",
}


# ============================================================
# 4) ESTILO VISUAL
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 19,
    "axes.labelsize": 23,
    "axes.titlesize": 24,
    "xtick.labelsize": 16,
    "ytick.labelsize": 17,
    "legend.fontsize": 13,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

CORES_METODO = {
    "RF_original_wide": "tab:blue",
    "Park": "tab:green",
}

CORES_DANO = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:red",
}


# ============================================================
# 5) FUNÇÕES BÁSICAS
# ============================================================

def salvar_fig(fig, nome_base, pasta=PASTA_GRAFICOS):
    os.makedirs(pasta, exist_ok=True)
    png = os.path.join(pasta, nome_base + ".png")
    fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")

    if SALVAR_PDF:
        pdf = os.path.join(pasta, nome_base + ".pdf")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white")
        print(f"✅ Salvo:\n{png}\n{pdf}")
    else:
        print(f"✅ Salvo:\n{png}")


def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both")


def format_temp(T):
    T = float(T)
    return str(int(T)) if T.is_integer() else f"{T:.1f}"


def format_faixa(fmin, fmax):
    return f"{fmin:.1f}–{fmax:.1f} kHz"


def safe_name(s):
    return str(s).replace(" ", "_").replace("–", "-").replace("/", "_").replace("+", "p")


def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            f_khz = f / 1e3
            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)
    if len(cols) == 0:
        return [], np.array([], dtype=float)
    order = np.argsort(freqs)
    cols = [cols[i] for i in order]
    freqs = np.array(freqs, dtype=float)[order]
    return cols, freqs


def moving_average(arr, win):
    arr = np.asarray(arr, dtype=float)
    win = int(win)
    if win <= 1:
        return arr.copy()
    if win % 2 == 0:
        win += 1
    if win >= len(arr):
        win = max(3, len(arr) // 5)
        if win % 2 == 0:
            win += 1
    if win <= 1 or win >= len(arr):
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win, dtype=float) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")
    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")
    return smooth


def add_extra_features_matrix(X):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    q10 = np.quantile(X, 0.10, axis=1, keepdims=True)
    q50 = np.quantile(X, 0.50, axis=1, keepdims=True)
    q90 = np.quantile(X, 0.90, axis=1, keepdims=True)
    return np.hstack([X, mu, sd, amp, q10, q50, q90])


def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)
    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18
    corr = num / den
    return float(1 - corr)


def curva_referencia_saudavel(df, fcols, ref_temp=REF_TEMP):
    df_sem = df[df["falha"] == 0].copy()
    if len(df_sem) == 0:
        raise ValueError("Não há amostras sem dano para montar referência.")

    temps_sem = np.asarray(sorted(df_sem["temperatura_c"].dropna().unique()), dtype=float)
    if len(temps_sem) == 0:
        raise ValueError("Não há temperaturas válidas no conjunto sem dano.")

    if np.any(np.isclose(temps_sem, ref_temp)):
        temp_usada = float(temps_sem[np.where(np.isclose(temps_sem, ref_temp))[0][0]])
    else:
        temp_usada = float(temps_sem[np.argmin(np.abs(temps_sem - ref_temp))])
        print(f"⚠️ Não achei sem dano em {ref_temp}°C. Usando referência saudável em {temp_usada}°C.")

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], temp_usada), fcols].to_numpy(float)
    return np.median(pool, axis=0), temp_usada


def col_indices(source_cols, target_cols):
    pos = {c: i for i, c in enumerate(source_cols)}
    return np.array([pos[c] for c in target_cols], dtype=int)


def make_reduced_indices(n_points, max_points=4500, step_min=1):
    n_points = int(n_points)
    if n_points <= max_points:
        return np.arange(n_points, dtype=int), 1

    step = int(np.ceil(n_points / float(max_points)))
    step = max(int(step_min), step)
    idx = np.arange(0, n_points, step, dtype=int)
    if idx[-1] != n_points - 1:
        idx = np.r_[idx, n_points - 1]
    return np.unique(idx), step


def interp_rows_to_full(y_reduced, idx_reduced, n_points):
    y_reduced = np.asarray(y_reduced, dtype=float)
    idx_reduced = np.asarray(idx_reduced, dtype=float)
    grid_full = np.arange(int(n_points), dtype=float)

    Y_full = np.empty((y_reduced.shape[0], int(n_points)), dtype=float)
    for i in range(y_reduced.shape[0]):
        Y_full[i] = np.interp(grid_full, idx_reduced, y_reduced[i])
    return Y_full


# ============================================================
# 6) COMPENSAÇÃO RF ORIGINAL E PARK EM FAIXA LARGA
# ============================================================

def compensar_rf_original_wide(df_comp_band, fcols_comp, y_ref_comp, mask_treino_sem_dano=None):
    cfg = RF_VARIANTS["RF_original_wide"]
    params = cfg["params"]
    smooth_win = int(cfg.get("smooth_win", SMOOTH_WIN_PADRAO))

    if mask_treino_sem_dano is None:
        mask_treino_sem_dano = (df_comp_band["falha"].to_numpy(int) == 0)
    else:
        mask_treino_sem_dano = np.asarray(mask_treino_sem_dano, dtype=bool)

    df_sem = df_comp_band.loc[mask_treino_sem_dano].copy()
    if len(df_sem) < 3:
        raise ValueError("Poucas amostras sem dano para treinar RF_original_wide.")

    X_sem = df_sem[fcols_comp].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    X_all = df_comp_band[fcols_comp].to_numpy(float)
    T_all = df_comp_band["temperatura_c"].to_numpy(float)

    n_points = len(fcols_comp)

    if USAR_RF_DOWNSAMPLE_SAIDA:
        idx_out, step_usado = make_reduced_indices(
            n_points,
            max_points=RF_MAX_OUTPUT_POINTS,
            step_min=RF_DOWNSAMPLE_STEP_MIN,
        )
    else:
        idx_out = np.arange(n_points, dtype=int)
        step_usado = 1

    X_in = np.hstack([
        add_extra_features_matrix(X_sem),
        T_sem.reshape(-1, 1),
    ])

    Y_target_reduced = y_ref_comp[None, idx_out] - X_sem[:, idx_out]

    if PRINT_PROGRESS_METHODS:
        print(
            f"      • RF original 25–60: treinando RF com {len(idx_out)}/{n_points} saídas "
            f"(step={step_usado})"
        )

    rf = RandomForestRegressor(**params)
    rf.fit(X_in, Y_target_reduced)

    X_all_in = np.hstack([
        add_extra_features_matrix(X_all),
        T_all.reshape(-1, 1),
    ])

    delta_pred_reduced = rf.predict(X_all_in)

    if len(idx_out) < n_points:
        delta_pred = interp_rows_to_full(delta_pred_reduced, idx_out, n_points)
    else:
        delta_pred = delta_pred_reduced

    Y_comp = X_all + delta_pred

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], smooth_win)

    df2 = df_comp_band.copy()
    df2[fcols_comp] = Y_comp
    return df2


def shift_interp(x, f, tau):
    f_shift = f + tau
    return np.interp(f, f_shift, x, left=x[0], right=x[-1])


def park_single(x, ref, fHz):
    x = np.asarray(x, dtype=float)
    ref = np.asarray(ref, dtype=float)
    fHz = np.asarray(fHz, dtype=float)

    n_points = len(fHz)
    if USAR_PARK_DOWNSAMPLE_FIT:
        idx_fit, step_fit = make_reduced_indices(n_points, max_points=PARK_FIT_MAX_POINTS, step_min=1)
    else:
        idx_fit = np.arange(n_points, dtype=int)
        step_fit = 1

    x_fit = x[idx_fit]
    ref_fit = ref[idx_fit]
    f_fit = fHz[idx_fit]

    df_band = f_fit[-1] - f_fit[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift_fit = shift_interp(x_fit, f_fit, tau)
        dS_fit = np.mean(ref_fit - x_shift_fit)
        err = np.sum((ref_fit - (x_shift_fit + dS_fit)) ** 2)
        if err < best_err:
            best_err = err
            best_tau = tau

    x_shift_full = shift_interp(x, fHz, best_tau)
    best_dS = np.mean(ref - x_shift_full)
    y = x_shift_full + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)
    return y


def compensar_park_wide(df_comp_band, fcols_comp, fHz_comp, y_ref_comp):
    X_all = df_comp_band[fcols_comp].to_numpy(float)
    Y = np.zeros_like(X_all)

    if PRINT_PROGRESS_METHODS:
        n_points = len(fcols_comp)
        idx_fit, step_fit = make_reduced_indices(n_points, max_points=PARK_FIT_MAX_POINTS, step_min=1)
        print(
            f"      • Park: buscando shift com {len(idx_fit)}/{n_points} pontos "
            f"(step={step_fit}) e aplicando na curva completa"
        )

    for i in range(len(X_all)):
        Y[i] = park_single(X_all[i], y_ref_comp, fHz_comp)

    df2 = df_comp_band.copy()
    df2[fcols_comp] = Y
    return df2


def calcular_metricas_df_window(df_comp_wide, fcols_window, y_ref_window):
    X = df_comp_wide[fcols_window].to_numpy(float)
    out = df_comp_wide[["temperatura_c", "falha"]].copy()
    out["RMSD"] = [rmsd(x, y_ref_window) for x in X]
    out["CCDM"] = [ccdm(x, y_ref_window) for x in X]
    return out


# ============================================================
# 7) PICOS, VALES E JANELAS
# ============================================================

def detectar_picos_vales(df_base):
    fcols_global, fHz_global = get_freq_columns(
        df_base,
        FREQ_PEAK_SEARCH_MIN_KHZ,
        FREQ_PEAK_SEARCH_MAX_KHZ,
    )

    if len(fcols_global) < 10:
        raise ValueError("Poucos pontos na região de busca de picos.")

    y_ref_global, temp_ref_usada = curva_referencia_saudavel(df_base, fcols_global, REF_TEMP)
    fkhz_global = fHz_global / 1e3

    win = 11 if len(y_ref_global) >= 15 else 5
    if win % 2 == 0:
        win += 1
    y_s = moving_average(y_ref_global, win)
    y_norm = (y_s - np.mean(y_s)) / (np.std(y_s) + 1e-18)

    candidatos = []

    if len(PEAK_CENTERS_MANUAL_KHZ) > 0:
        for c in PEAK_CENTERS_MANUAL_KHZ:
            candidatos.append({"centro_khz": float(c), "tipo": "manual", "score": 999.0})
    else:
        try:
            from scipy.signal import find_peaks
            dfreq = float(np.median(np.diff(fkhz_global))) if len(fkhz_global) > 2 else 0.1
            min_dist_pts = max(1, int(round(MIN_DIST_ENTRE_PICOS_KHZ / max(dfreq, 1e-9))))

            peaks, prop_p = find_peaks(y_norm, distance=min_dist_pts, prominence=0.10)
            valleys, prop_v = find_peaks(-y_norm, distance=min_dist_pts, prominence=0.10)

            for idx, prom in zip(peaks, prop_p.get("prominences", np.ones(len(peaks)))):
                candidatos.append({"centro_khz": float(fkhz_global[idx]), "tipo": "pico", "score": float(prom)})
            for idx, prom in zip(valleys, prop_v.get("prominences", np.ones(len(valleys)))):
                candidatos.append({"centro_khz": float(fkhz_global[idx]), "tipo": "vale", "score": float(prom)})

        except Exception:
            dy = np.diff(y_norm)
            sign = np.sign(dy)
            for i in range(1, len(sign)):
                if sign[i - 1] > 0 and sign[i] < 0:
                    local = y_norm[max(0, i - 10):min(len(y_norm), i + 11)]
                    score = abs(y_norm[i] - np.median(local))
                    candidatos.append({"centro_khz": float(fkhz_global[i]), "tipo": "pico", "score": float(score)})
                if sign[i - 1] < 0 and sign[i] > 0:
                    local = y_norm[max(0, i - 10):min(len(y_norm), i + 11)]
                    score = abs(y_norm[i] - np.median(local))
                    candidatos.append({"centro_khz": float(fkhz_global[i]), "tipo": "vale", "score": float(score)})

    if len(candidatos) == 0:
        raise RuntimeError("Não consegui detectar picos/vales. Use PEAK_CENTERS_MANUAL_KHZ.")

    cand = pd.DataFrame(candidatos).sort_values("score", ascending=False).reset_index(drop=True)

    selecionados = []
    for _, r in cand.iterrows():
        c = float(r["centro_khz"])
        if c < FREQ_PEAK_SEARCH_MIN_KHZ or c > FREQ_PEAK_SEARCH_MAX_KHZ:
            continue
        if all(abs(c - s["centro_khz"]) >= MIN_DIST_ENTRE_PICOS_KHZ for s in selecionados):
            selecionados.append(r.to_dict())
        if len(selecionados) >= TOP_N_PEAK_WINDOWS:
            break

    half = PEAK_WINDOW_WIDTH_KHZ / 2.0
    rows = []

    for r in selecionados:
        centro = float(r["centro_khz"])
        fmin = max(FREQ_PEAK_SEARCH_MIN_KHZ, centro - half)
        fmax = min(FREQ_PEAK_SEARCH_MAX_KHZ, centro + half)
        fcols, _ = get_freq_columns(df_base, fmin, fmax)
        if len(fcols) < MIN_FREQ_POINTS_WINDOW:
            continue
        rows.append({
            "centro_khz": centro,
            "faixa_min_khz": float(fmin),
            "faixa_max_khz": float(fmax),
            "largura_khz": float(fmax - fmin),
            "tipo_extremo": r["tipo"],
            "score_extremo": float(r["score"]),
            "n_freq_points": int(len(fcols)),
            "faixa_label": format_faixa(fmin, fmax),
        })

    if INCLUIR_JANELA_J2_MANUAL:
        fmin, fmax = JANELA_J2_MANUAL
        fcols, _ = get_freq_columns(df_base, fmin, fmax)
        if len(fcols) >= MIN_FREQ_POINTS_WINDOW:
            rows.append({
                "centro_khz": 0.5 * (fmin + fmax),
                "faixa_min_khz": float(fmin),
                "faixa_max_khz": float(fmax),
                "largura_khz": float(fmax - fmin),
                "tipo_extremo": "manual_J2",
                "score_extremo": 9999.0,
                "n_freq_points": int(len(fcols)),
                "faixa_label": format_faixa(fmin, fmax),
            })

    df_windows = pd.DataFrame(rows)
    if len(df_windows) == 0:
        raise RuntimeError("Nenhuma janela válida foi gerada.")

    df_windows["fmin_round"] = df_windows["faixa_min_khz"].round(3)
    df_windows["fmax_round"] = df_windows["faixa_max_khz"].round(3)
    df_windows = df_windows.sort_values(["score_extremo"], ascending=False)
    df_windows = df_windows.drop_duplicates(["fmin_round", "fmax_round"], keep="first")
    df_windows = df_windows.drop(columns=["fmin_round", "fmax_round"])
    df_windows = df_windows.sort_values("centro_khz").reset_index(drop=True)
    df_windows["janela_id"] = np.arange(len(df_windows))

    df_windows.to_csv(os.path.join(PASTA_CSV, "janelas_picos_selecionadas.csv"), index=False)
    return df_windows, fkhz_global, y_ref_global, temp_ref_usada


def plot_janelas_picos(df_windows, fkhz_global, y_ref_global, temp_ref_usada):
    fig, ax = plt.subplots(figsize=(17, 8), dpi=300)
    ax.plot(fkhz_global, y_ref_global, color="black", linewidth=1.8, label=f"Referência saudável {format_temp(temp_ref_usada)}°C")

    for _, r in df_windows.iterrows():
        ax.axvspan(r["faixa_min_khz"], r["faixa_max_khz"], alpha=0.16)
        ax.axvline(r["centro_khz"], color="black", linewidth=0.8, alpha=0.35)
        ax.text(
            r["centro_khz"],
            np.nanmax(y_ref_global),
            f"J{int(r['janela_id'])}\n{r['centro_khz']:.1f}",
            ha="center",
            va="top",
            fontsize=10,
        )

    ax.axvspan(COMP_FREQ_MIN_KHZ, COMP_FREQ_MAX_KHZ, alpha=0.06, color="gray", label="Faixa usada pelo RF/Park")
    ax.set_xlabel("Frequência (kHz)")
    ax.set_ylabel("Impedância")
    ax.set_title("Janelas analisadas dentro da faixa larga usada pelo RF original/Park", pad=14)
    ax.legend(frameon=True, loc="best")
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, "janelas_pequenas_com_RF_original_25_60")
    plt.show()


# ============================================================
# 8) CLASSIFICAÇÃO
# ============================================================

def get_valid_temperatures_for_loto(df):
    temps = []
    for T in sorted(df["temperatura_c"].dropna().unique()):
        ok = True
        for d in DANOS:
            if not np.any(np.isclose(df["temperatura_c"], T) & (df["falha"] == d)):
                ok = False
                break
        if ok:
            temps.append(float(T))
    return temps


def build_features(df_comp_wide, fcols_window, df_metrics_window, feature_set):
    if feature_set == "metricas_RMSD_CCDM":
        return df_metrics_window[["RMSD", "CCDM"]].to_numpy(float)

    if feature_set == "curva_e_metricas":
        X_curve = df_comp_wide[fcols_window].to_numpy(float)
        X_metric = df_metrics_window[["RMSD", "CCDM"]].to_numpy(float)
        return np.hstack([X_curve, X_metric])

    raise ValueError(f"feature_set desconhecido: {feature_set}")


def make_classifier():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=0,
        )),
    ])


def preparar_compensacoes_largas_para_fold(df_comp_band, fcols_comp, fHz_comp, T_test):
    test_mask = np.isclose(df_comp_band["temperatura_c"].to_numpy(float), float(T_test))
    train_mask = ~test_mask

    if USAR_REFERENCIA_GLOBAL:
        y_ref_comp, temp_ref_usada = curva_referencia_saudavel(df_comp_band, fcols_comp, REF_TEMP)
    else:
        y_ref_comp, temp_ref_usada = curva_referencia_saudavel(df_comp_band.loc[train_mask].copy(), fcols_comp, REF_TEMP)

    if STRICT_LOTO_COMPENSATION:
        mask_treino_rf = train_mask & (df_comp_band["falha"].to_numpy(int) == 0)
    else:
        mask_treino_rf = (df_comp_band["falha"].to_numpy(int) == 0)

    dados_metodos = {}

    df_rf = compensar_rf_original_wide(
        df_comp_band,
        fcols_comp,
        y_ref_comp,
        mask_treino_sem_dano=mask_treino_rf,
    )
    dados_metodos["RF_original_wide"] = df_rf

    df_pk = compensar_park_wide(df_comp_band, fcols_comp, fHz_comp, y_ref_comp)
    dados_metodos["Park"] = df_pk

    return dados_metodos, y_ref_comp, temp_ref_usada, test_mask, train_mask


def avaliar_todas_janelas(df_base, df_windows):
    fcols_comp, fHz_comp = get_freq_columns(df_base, COMP_FREQ_MIN_KHZ, COMP_FREQ_MAX_KHZ)
    if len(fcols_comp) < 20:
        raise ValueError(f"Poucos pontos na faixa larga {COMP_FREQ_MIN_KHZ}-{COMP_FREQ_MAX_KHZ} kHz.")

    df_comp_band = df_base[["temperatura_c", "falha"] + fcols_comp].copy().reset_index(drop=True)

    temps_loto = get_valid_temperatures_for_loto(df_comp_band)
    if len(temps_loto) < 2:
        raise ValueError("Poucas temperaturas válidas para LOTO.")

    print(f"\n✅ RF/Park agirão em {COMP_FREQ_MIN_KHZ:.1f}–{COMP_FREQ_MAX_KHZ:.1f} kHz | pontos = {len(fcols_comp)}")
    if USAR_RF_DOWNSAMPLE_SAIDA:
        idx_tmp, step_tmp = make_reduced_indices(len(fcols_comp), RF_MAX_OUTPUT_POINTS, RF_DOWNSAMPLE_STEP_MIN)
        print(f"✅ RF usará {len(idx_tmp)} pontos de saída e interpolará para {len(fcols_comp)} pontos (step={step_tmp})")
    if USAR_PARK_DOWNSAMPLE_FIT:
        idx_tmp, step_tmp = make_reduced_indices(len(fcols_comp), PARK_FIT_MAX_POINTS, 1)
        print(f"✅ Park buscará shift em {len(idx_tmp)} pontos e aplicará na curva completa (step={step_tmp})")
    print(f"✅ Análise/classificação será feita em {len(df_windows)} janelas pequenas.")

    y_multi = df_comp_band["falha"].to_numpy(int)
    y_bin = (y_multi > 0).astype(int)

    resultados_multi = []
    resultados_bin = []
    metricas_teste_rows = []
    pred_multi_rows = []
    pred_bin_rows = []

    for T_test in temps_loto:
        print("\n" + "-" * 100)
        print(f"🔹 Fold LOTO: testando T = {format_temp(T_test)}°C | RF original treinado em 25–60 kHz")

        test_mask_global = np.isclose(df_comp_band["temperatura_c"].to_numpy(float), T_test)
        train_mask_global = ~test_mask_global

        y_train_multi = y_multi[train_mask_global]
        y_test_multi = y_multi[test_mask_global]
        y_train_bin = y_bin[train_mask_global]
        y_test_bin = y_bin[test_mask_global]

        if len(np.unique(y_train_multi)) < 3 or len(np.unique(y_test_multi)) < 2:
            continue
        if len(np.unique(y_train_bin)) < 2 or len(np.unique(y_test_bin)) < 2:
            continue

        dados_metodos, y_ref_comp, temp_ref_usada, test_mask, train_mask = preparar_compensacoes_largas_para_fold(
            df_comp_band,
            fcols_comp,
            fHz_comp,
            T_test,
        )

        for _, row_window in df_windows.iterrows():
            fmin = float(row_window["faixa_min_khz"])
            fmax = float(row_window["faixa_max_khz"])
            janela_id = int(row_window["janela_id"])
            faixa_label = str(row_window["faixa_label"])

            fcols_window, _ = get_freq_columns(df_base, fmin, fmax)
            if len(fcols_window) < MIN_FREQ_POINTS_WINDOW:
                continue

            missing = [c for c in fcols_window if c not in fcols_comp]
            if missing:
                print(f"⚠️ Janela {faixa_label} tem colunas fora da faixa larga. Pulando.")
                continue

            idx_win = col_indices(fcols_comp, fcols_window)
            y_ref_window = y_ref_comp[idx_win]

            for metodo, df_comp_wide in dados_metodos.items():
                df_met_window = calcular_metricas_df_window(df_comp_wide, fcols_window, y_ref_window)

                df_test_met = df_met_window.loc[test_mask].copy()
                for idx_local, r in df_test_met.iterrows():
                    metricas_teste_rows.append({
                        "janela_id": janela_id,
                        "faixa_min_khz": fmin,
                        "faixa_max_khz": fmax,
                        "faixa_label": faixa_label,
                        "centro_khz": float(row_window["centro_khz"]),
                        "tipo_extremo": row_window["tipo_extremo"],
                        "metodo": metodo,
                        "metodo_nome": NOME_METODO.get(metodo, metodo),
                        "fold_test_temp": float(T_test),
                        "temperatura_c": float(r["temperatura_c"]),
                        "falha": int(r["falha"]),
                        "classe_binaria": int(int(r["falha"]) > 0),
                        "RMSD": float(r["RMSD"]),
                        "CCDM": float(r["CCDM"]),
                        "temp_ref_usada": float(temp_ref_usada),
                        "comp_freq_min_khz": COMP_FREQ_MIN_KHZ,
                        "comp_freq_max_khz": COMP_FREQ_MAX_KHZ,
                    })

                for fs in FEATURE_SETS:
                    X_feat = build_features(df_comp_wide, fcols_window, df_met_window, fs)

                    clf_multi = make_classifier()
                    clf_multi.fit(X_feat[train_mask], y_train_multi)
                    pred_multi = clf_multi.predict(X_feat[test_mask])

                    acc = accuracy_score(y_test_multi, pred_multi)
                    bacc = balanced_accuracy_score(y_test_multi, pred_multi)
                    mf1 = f1_score(y_test_multi, pred_multi, average="macro", labels=LABELS_MULTI, zero_division=0)
                    f1_por_dano = f1_score(y_test_multi, pred_multi, average=None, labels=LABELS_MULTI, zero_division=0)

                    resultados_multi.append({
                        "janela_id": janela_id,
                        "faixa_min_khz": fmin,
                        "faixa_max_khz": fmax,
                        "faixa_label": faixa_label,
                        "centro_khz": float(row_window["centro_khz"]),
                        "tipo_extremo": row_window["tipo_extremo"],
                        "metodo": metodo,
                        "metodo_nome": NOME_METODO.get(metodo, metodo),
                        "feature_set": fs,
                        "test_temp": float(T_test),
                        "accuracy": float(acc),
                        "balanced_accuracy": float(bacc),
                        "macro_f1": float(mf1),
                        "f1_dano0": float(f1_por_dano[0]),
                        "f1_dano1": float(f1_por_dano[1]),
                        "f1_dano2": float(f1_por_dano[2]),
                        "n_test": int(len(y_test_multi)),
                    })

                    for yt, yp in zip(y_test_multi, pred_multi):
                        pred_multi_rows.append({
                            "janela_id": janela_id,
                            "faixa_label": faixa_label,
                            "metodo": metodo,
                            "metodo_nome": NOME_METODO.get(metodo, metodo),
                            "feature_set": fs,
                            "test_temp": float(T_test),
                            "y_true": int(yt),
                            "y_pred": int(yp),
                        })

                    clf_bin = make_classifier()
                    clf_bin.fit(X_feat[train_mask], y_train_bin)
                    pred_bin = clf_bin.predict(X_feat[test_mask])

                    acc_bin = accuracy_score(y_test_bin, pred_bin)
                    bacc_bin = balanced_accuracy_score(y_test_bin, pred_bin)
                    mf1_bin = f1_score(y_test_bin, pred_bin, average="macro", labels=LABELS_BIN, zero_division=0)
                    recall_dano = recall_score(y_test_bin, pred_bin, pos_label=1, zero_division=0)
                    recall_sem = recall_score(y_test_bin, pred_bin, pos_label=0, zero_division=0)
                    precision_dano = precision_score(y_test_bin, pred_bin, pos_label=1, zero_division=0)

                    cm_bin = confusion_matrix(y_test_bin, pred_bin, labels=LABELS_BIN)
                    falso_saudavel = int(cm_bin[1, 0])
                    n_dano_real = int(cm_bin[1, :].sum())
                    taxa_falso_saudavel = falso_saudavel / max(n_dano_real, 1)

                    resultados_bin.append({
                        "janela_id": janela_id,
                        "faixa_min_khz": fmin,
                        "faixa_max_khz": fmax,
                        "faixa_label": faixa_label,
                        "centro_khz": float(row_window["centro_khz"]),
                        "tipo_extremo": row_window["tipo_extremo"],
                        "metodo": metodo,
                        "metodo_nome": NOME_METODO.get(metodo, metodo),
                        "feature_set": fs,
                        "test_temp": float(T_test),
                        "accuracy_bin": float(acc_bin),
                        "balanced_accuracy_bin": float(bacc_bin),
                        "macro_f1_bin": float(mf1_bin),
                        "recall_dano": float(recall_dano),
                        "recall_sem_dano": float(recall_sem),
                        "precision_dano": float(precision_dano),
                        "falso_saudavel_count": int(falso_saudavel),
                        "taxa_falso_saudavel": float(taxa_falso_saudavel),
                        "n_test": int(len(y_test_bin)),
                        "n_dano_real": int(n_dano_real),
                    })

                    for yt, yp in zip(y_test_bin, pred_bin):
                        pred_bin_rows.append({
                            "janela_id": janela_id,
                            "faixa_label": faixa_label,
                            "metodo": metodo,
                            "metodo_nome": NOME_METODO.get(metodo, metodo),
                            "feature_set": fs,
                            "test_temp": float(T_test),
                            "y_true_bin": int(yt),
                            "y_pred_bin": int(yp),
                        })

    return {
        "folds_multi": pd.DataFrame(resultados_multi),
        "folds_bin": pd.DataFrame(resultados_bin),
        "metricas_teste": pd.DataFrame(metricas_teste_rows),
        "pred_multi": pd.DataFrame(pred_multi_rows),
        "pred_bin": pd.DataFrame(pred_bin_rows),
    }


# ============================================================
# 9) RESUMOS
# ============================================================

def resumir_multiclasse(df_folds):
    if len(df_folds) == 0:
        return pd.DataFrame()
    return (
        df_folds
        .groupby(["janela_id", "faixa_min_khz", "faixa_max_khz", "faixa_label", "centro_khz", "tipo_extremo", "metodo", "metodo_nome", "feature_set"], as_index=False)
        .agg(
            accuracy_medio=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
            balanced_accuracy_medio=("balanced_accuracy", "mean"),
            balanced_accuracy_std=("balanced_accuracy", "std"),
            macro_f1_medio=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            f1_dano0_medio=("f1_dano0", "mean"),
            f1_dano1_medio=("f1_dano1", "mean"),
            f1_dano2_medio=("f1_dano2", "mean"),
            n_folds=("test_temp", "nunique"),
        )
        .sort_values("macro_f1_medio", ascending=False)
        .reset_index(drop=True)
    )


def resumir_binario(df_folds_bin):
    if len(df_folds_bin) == 0:
        return pd.DataFrame()
    return (
        df_folds_bin
        .groupby(["janela_id", "faixa_min_khz", "faixa_max_khz", "faixa_label", "centro_khz", "tipo_extremo", "metodo", "metodo_nome", "feature_set"], as_index=False)
        .agg(
            accuracy_bin_medio=("accuracy_bin", "mean"),
            balanced_accuracy_bin_medio=("balanced_accuracy_bin", "mean"),
            macro_f1_bin_medio=("macro_f1_bin", "mean"),
            recall_dano_medio=("recall_dano", "mean"),
            recall_sem_dano_medio=("recall_sem_dano", "mean"),
            precision_dano_medio=("precision_dano", "mean"),
            falso_saudavel_total=("falso_saudavel_count", "sum"),
            taxa_falso_saudavel_media=("taxa_falso_saudavel", "mean"),
            n_folds=("test_temp", "nunique"),
        )
        .sort_values(["recall_dano_medio", "taxa_falso_saudavel_media", "macro_f1_bin_medio"], ascending=[False, True, False])
        .reset_index(drop=True)
    )


# ============================================================
# 10) GRÁFICOS
# ============================================================

def plot_ranking_multiclasse(df_resumo):
    sub = df_resumo[df_resumo["feature_set"] == FEATURE_SET_PREFERIDO].copy()
    if len(sub) == 0:
        return None
    top = sub.sort_values("macro_f1_medio", ascending=False).copy()
    top["label"] = top.apply(lambda r: f"J{int(r['janela_id'])} {r['faixa_label']} — {r['metodo_nome']}", axis=1)
    top = top.iloc[::-1]

    fig, ax = plt.subplots(figsize=(17, 9), dpi=300)
    y = np.arange(len(top))
    colors = [CORES_METODO.get(m, None) for m in top["metodo"]]
    ax.barh(y, top["macro_f1_medio"], xerr=top["macro_f1_std"].fillna(0), color=colors, edgecolor="black", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(top["label"], fontsize=12)
    ax.set_xlabel("Macro-F1 médio — D0/D1/D2")
    ax.set_title(f"Ranking multiclasse — RF original/Park em 25–60 kHz — {FEATURE_SET_PREFERIDO}", pad=14)
    ax.set_xlim(0, min(1.05, max(0.2, top["macro_f1_medio"].max() + 0.15)))
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, "ranking_multiclasse_macroF1_RF_original_vs_Park")
    plt.show()
    return fig


def plot_ranking_binario(df_resumo_bin):
    sub = df_resumo_bin[df_resumo_bin["feature_set"] == FEATURE_SET_PREFERIDO].copy()
    if len(sub) == 0:
        return None

    top = sub.sort_values(["recall_dano_medio", "taxa_falso_saudavel_media", "macro_f1_bin_medio"], ascending=[False, True, False]).copy()
    top["label"] = top.apply(lambda r: f"J{int(r['janela_id'])} {r['faixa_label']} — {r['metodo_nome']}", axis=1)
    top = top.iloc[::-1]

    fig, ax = plt.subplots(figsize=(17, 9), dpi=300)
    y = np.arange(len(top))
    colors = [CORES_METODO.get(m, None) for m in top["metodo"]]
    ax.barh(y, top["recall_dano_medio"], color=colors, edgecolor="black", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(top["label"], fontsize=12)
    ax.set_xlabel("Recall médio da classe com dano")
    ax.set_title("Ranking binário — RF original/Park em 25–60 kHz", pad=14)
    ax.set_xlim(0, 1.05)
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, "ranking_binario_recall_dano_RF_original_vs_Park")
    plt.show()

    fig, ax = plt.subplots(figsize=(17, 9), dpi=300)
    ax.barh(y, top["taxa_falso_saudavel_media"], color=colors, edgecolor="black", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(top["label"], fontsize=12)
    ax.set_xlabel("Taxa média de falso saudável")
    ax.set_title("Erro crítico em SHM — dano real classificado como sem dano", pad=14)
    ax.set_xlim(0, max(0.05, min(1.05, top["taxa_falso_saudavel_media"].max() + 0.1)))
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, "ranking_binario_falso_saudavel_RF_original_vs_Park")
    plt.show()


def plot_scatter_binario(df_resumo_bin):
    sub = df_resumo_bin[df_resumo_bin["feature_set"] == FEATURE_SET_PREFERIDO].copy()
    if len(sub) == 0:
        return None

    fig, ax = plt.subplots(figsize=(12, 9), dpi=300)
    for metodo, gm in sub.groupby("metodo"):
        ax.scatter(
            gm["recall_sem_dano_medio"],
            gm["recall_dano_medio"],
            s=110,
            alpha=0.85,
            edgecolor="black",
            linewidth=0.7,
            color=CORES_METODO.get(metodo, None),
            label=NOME_METODO.get(metodo, metodo),
        )
        for _, r in gm.iterrows():
            ax.text(r["recall_sem_dano_medio"] + 0.008, r["recall_dano_medio"] + 0.008, f"J{int(r['janela_id'])}", fontsize=9)

    ax.set_xlabel("Recall sem dano / especificidade")
    ax.set_ylabel("Recall com dano / sensibilidade")
    ax.set_title("Trade-off binário: proteger contra falso saudável", pad=14)
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(frameon=True, fontsize=10, loc="lower left")
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, "scatter_binario_recall_sem_dano_vs_recall_dano")
    plt.show()


def plot_confusion_from_predictions(df_pred, janela_id, metodo, feature_set, binario=False):
    sub = df_pred[(df_pred["janela_id"] == janela_id) & (df_pred["metodo"] == metodo) & (df_pred["feature_set"] == feature_set)].copy()
    if len(sub) == 0:
        print(f"⚠️ Sem predições para matriz: J{janela_id}, {metodo}, {feature_set}")
        return None

    if binario:
        y_true = sub["y_true_bin"].to_numpy(int)
        y_pred = sub["y_pred_bin"].to_numpy(int)
        labels = LABELS_BIN
        labels_txt = LABELS_BIN_TXT
        titulo_tipo = "binária"
    else:
        y_true = sub["y_true"].to_numpy(int)
        y_pred = sub["y_pred"].to_numpy(int)
        labels = LABELS_MULTI
        labels_txt = LABELS_MULTI_TXT
        titulo_tipo = "multiclasse"

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(8.5, 7.2), dpi=300)
    im = ax.imshow(cm, aspect="auto")
    ax.set_xticks(np.arange(len(labels_txt)))
    ax.set_xticklabels(labels_txt)
    ax.set_yticks(np.arange(len(labels_txt)))
    ax.set_yticklabels(labels_txt)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusão {titulo_tipo} — J{janela_id} — {NOME_METODO.get(metodo, metodo)}", pad=14)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center", fontsize=26, color="black")

    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=14)
    fig.tight_layout()
    nome = f"matriz_confusao_{'binaria' if binario else 'multi'}_J{janela_id}_{safe_name(metodo)}_{safe_name(feature_set)}"
    salvar_fig(fig, nome, pasta=PASTA_CONFUSAO)
    plt.show()
    return fig


def plot_metricas_por_temperatura(df_metricas, janela_id, metodos_escolhidos):
    sub = df_metricas[(df_metricas["janela_id"] == janela_id) & (df_metricas["metodo"].isin(metodos_escolhidos))].copy()
    if len(sub) == 0:
        return None

    for metrica in ["RMSD", "CCDM"]:
        fig, axes = plt.subplots(1, len(metodos_escolhidos), figsize=(8 * len(metodos_escolhidos), 7), dpi=300, sharey=False)
        if len(metodos_escolhidos) == 1:
            axes = [axes]

        for ax, metodo in zip(axes, metodos_escolhidos):
            sm = sub[sub["metodo"] == metodo]
            if len(sm) == 0:
                continue

            for d in DANOS:
                g = (
                    sm[sm["falha"] == d]
                    .groupby("temperatura_c", as_index=False)[metrica]
                    .mean()
                    .sort_values("temperatura_c")
                )
                ax.plot(g["temperatura_c"], g[metrica], marker="o", linewidth=2.8, markersize=8, label=f"Dano {d}", color=CORES_DANO[d])

            faixa = sm["faixa_label"].iloc[0]
            ax.axvline(REF_TEMP, color="black", linestyle="--", linewidth=1.2, label=f"Ref. {REF_TEMP:.0f}°C")
            ax.set_xlabel("Temperatura (°C)")
            ax.set_ylabel(metrica)
            ax.set_title(f"{NOME_METODO.get(metodo, metodo)}\nJ{janela_id} ({faixa})", pad=14)
            ax.legend(frameon=True, loc="best")
            estilo_eixos(ax)

        fig.suptitle(f"{metrica} por temperatura — RF original vs Park", fontsize=26, y=1.02)
        fig.tight_layout()
        salvar_fig(fig, f"RF_original_vs_Park_J{janela_id}_{metrica}_por_temperatura", pasta=PASTA_METRICAS)
        plt.show()


def escolher_temperatura_disponivel(df, T_alvo):
    temps_validas = []
    for T in sorted(df["temperatura_c"].dropna().unique()):
        ok = True
        for d in DANOS:
            if not np.any(np.isclose(df["temperatura_c"], T) & (df["falha"] == d)):
                ok = False
                break
        if ok:
            temps_validas.append(float(T))
    if len(temps_validas) == 0:
        raise ValueError("Nenhuma temperatura possui os três danos.")
    arr = np.asarray(temps_validas)
    return float(arr[np.argmin(np.abs(arr - T_alvo))])


def plot_curvas_exemplo(df_base, row_window, T_alvo=TEMP_EXEMPLO_CURVAS):
    fcols_comp, fHz_comp = get_freq_columns(df_base, COMP_FREQ_MIN_KHZ, COMP_FREQ_MAX_KHZ)
    df_comp_band = df_base[["temperatura_c", "falha"] + fcols_comp].copy().reset_index(drop=True)
    y_ref_comp, temp_ref_usada = curva_referencia_saudavel(df_comp_band, fcols_comp, REF_TEMP)

    fmin = float(row_window["faixa_min_khz"])
    fmax = float(row_window["faixa_max_khz"])
    janela_id = int(row_window["janela_id"])
    fcols_window, fHz_window = get_freq_columns(df_base, fmin, fmax)
    fkhz = fHz_window / 1e3
    idx_win = col_indices(fcols_comp, fcols_window)
    y_ref_window = y_ref_comp[idx_win]

    # Para visualização, treina compensadores com todos os saudáveis em 25–60 kHz.
    comps = {}
    comps["RF_original_wide"] = compensar_rf_original_wide(
        df_comp_band,
        fcols_comp,
        y_ref_comp,
        mask_treino_sem_dano=(df_comp_band["falha"].to_numpy(int) == 0),
    )
    comps["Park"] = compensar_park_wide(df_comp_band, fcols_comp, fHz_comp, y_ref_comp)

    T = escolher_temperatura_disponivel(df_comp_band, T_alvo)

    fig, axes = plt.subplots(1, len(DANOS), figsize=(8 * len(DANOS), 7.5), dpi=300, sharey=True)
    if len(DANOS) == 1:
        axes = [axes]

    for ax, d in zip(axes, DANOS):
        mask = np.isclose(df_comp_band["temperatura_c"], T) & (df_comp_band["falha"] == d)
        idxs = np.where(mask.to_numpy())[0]
        if len(idxs) == 0:
            continue
        idx = int(idxs[0])

        y_orig_window = df_comp_band.iloc[idx][fcols_window].to_numpy(float)
        ax.plot(fkhz, y_ref_window, "--", color="black", linewidth=2.0, label=f"Referência {format_temp(temp_ref_usada)}°C")
        ax.plot(fkhz, y_orig_window, color="lightcoral", linewidth=1.8, alpha=0.75, label=f"Original {format_temp(T)}°C")

        for metodo, dfc in comps.items():
            y_comp_window = dfc.iloc[idx][fcols_window].to_numpy(float)
            ax.plot(fkhz, y_comp_window, linewidth=2.2, color=CORES_METODO.get(metodo, None), label=NOME_METODO.get(metodo, metodo))

        ax.set_title(f"Dano {d}")
        ax.set_xlabel("Frequência (kHz)")
        estilo_eixos(ax)

    axes[0].set_ylabel("Impedância")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), frameon=True, fontsize=12, bbox_to_anchor=(0.5, -0.07))
    fig.suptitle(
        f"Curvas exemplo — RF original/Park em 25–60 kHz — análise J{janela_id} ({format_faixa(fmin, fmax)}) — T = {format_temp(T)}°C",
        fontsize=25,
        y=1.02,
    )
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    salvar_fig(fig, f"curvas_exemplo_RForiginal_vs_Park_J{janela_id}_T{format_temp(T).replace('-', 'm')}", pasta=PASTA_CURVAS)
    plt.show()


# ============================================================
# 11) EXECUÇÃO PRINCIPAL
# ============================================================

def executar_teste():
    t0_total = time.time()
    print("=" * 100)
    print("TESTE — RF ORIGINAL/PARK AGINDO EM 25–60 kHz E ANÁLISE EM JANELAS PEQUENAS")
    print("=" * 100)

    print("\n🔹 Carregando base...")
    df_base = pd.read_pickle(ARQ_BASE).reset_index(drop=True)

    required = {"temperatura_c", "falha"}
    missing = required - set(df_base.columns)
    if missing:
        raise ValueError(f"A base está sem colunas obrigatórias: {missing}")

    df_base["temperatura_c"] = pd.to_numeric(df_base["temperatura_c"], errors="coerce")
    df_base["falha"] = pd.to_numeric(df_base["falha"], errors="coerce").astype(int)

    print(f"Total de amostras: {len(df_base)}")
    print(f"Temperaturas: {sorted(df_base['temperatura_c'].dropna().unique())}")
    print(f"Danos: {sorted(df_base['falha'].dropna().unique())}")

    fcols_comp, _ = get_freq_columns(df_base, COMP_FREQ_MIN_KHZ, COMP_FREQ_MAX_KHZ)
    print(f"\n✅ Faixa de ação do RF/Park: {COMP_FREQ_MIN_KHZ:.1f}–{COMP_FREQ_MAX_KHZ:.1f} kHz | pontos = {len(fcols_comp)}")
    if USAR_RF_DOWNSAMPLE_SAIDA:
        idx_tmp, step_tmp = make_reduced_indices(len(fcols_comp), RF_MAX_OUTPUT_POINTS, RF_DOWNSAMPLE_STEP_MIN)
        print(f"✅ RF otimizado: saída reduzida para {len(idx_tmp)} pontos, depois interpolada para {len(fcols_comp)} pontos (step={step_tmp})")
    if USAR_PARK_DOWNSAMPLE_FIT:
        idx_tmp, step_tmp = make_reduced_indices(len(fcols_comp), PARK_FIT_MAX_POINTS, 1)
        print(f"✅ Park otimizado: busca do shift em {len(idx_tmp)} pontos e aplicação na curva completa (step={step_tmp})")

    print("\n🔹 Detectando janelas pequenas perto de picos/vales...")
    df_windows, fkhz_global, y_ref_global, temp_ref_usada = detectar_picos_vales(df_base)
    print(df_windows[["janela_id", "faixa_label", "centro_khz", "tipo_extremo", "n_freq_points"]].to_string(index=False))
    plot_janelas_picos(df_windows, fkhz_global, y_ref_global, temp_ref_usada)

    print("\n🔹 Avaliando folds e janelas...")
    out = avaliar_todas_janelas(df_base, df_windows)

    df_multi = out["folds_multi"]
    df_bin = out["folds_bin"]
    df_metricas = out["metricas_teste"]
    df_pred_multi = out["pred_multi"]
    df_pred_bin = out["pred_bin"]

    if len(df_multi) == 0:
        raise RuntimeError("Nenhum resultado multiclasse foi gerado.")

    df_resumo_multi = resumir_multiclasse(df_multi)
    df_resumo_bin = resumir_binario(df_bin)

    arquivos = {
        "resultados_folds_multiclasse.csv": df_multi,
        "resultados_folds_binario.csv": df_bin,
        "resumo_multiclasse_por_janela.csv": df_resumo_multi,
        "resumo_binario_por_janela.csv": df_resumo_bin,
        "metricas_teste_por_amostra.csv": df_metricas,
        "predicoes_multiclasse.csv": df_pred_multi,
        "predicoes_binario.csv": df_pred_bin,
    }

    for nome, df_ in arquivos.items():
        if df_ is not None and len(df_) > 0:
            path = os.path.join(PASTA_CSV, nome)
            df_.to_csv(path, index=False)
            print(f"✅ CSV salvo: {path}")

    plot_ranking_multiclasse(df_resumo_multi)
    if len(df_resumo_bin) > 0:
        plot_ranking_binario(df_resumo_bin)
        plot_scatter_binario(df_resumo_bin)

    pref_multi = df_resumo_multi[df_resumo_multi["feature_set"] == FEATURE_SET_PREFERIDO].sort_values("macro_f1_medio", ascending=False)
    print("\n🏆 Top multiclasse — feature_set preferido:")
    print(pref_multi[[
        "janela_id", "faixa_label", "metodo_nome", "macro_f1_medio", "balanced_accuracy_medio",
        "f1_dano0_medio", "f1_dano1_medio", "f1_dano2_medio",
    ]].to_string(index=False))

    best_multi = pref_multi.iloc[0]
    best_multi_j = int(best_multi["janela_id"])
    best_multi_metodo = str(best_multi["metodo"])

    print("\n🏆 Melhor geral multiclasse:")
    print(best_multi.to_string())

    if len(df_resumo_bin) > 0:
        pref_bin = df_resumo_bin[df_resumo_bin["feature_set"] == FEATURE_SET_PREFERIDO].sort_values(
            ["recall_dano_medio", "taxa_falso_saudavel_media", "macro_f1_bin_medio"],
            ascending=[False, True, False],
        )
        print("\n🏆 Top binário — sem dano vs com dano:")
        print(pref_bin[[
            "janela_id", "faixa_label", "metodo_nome", "recall_dano_medio", "recall_sem_dano_medio",
            "macro_f1_bin_medio", "falso_saudavel_total", "taxa_falso_saudavel_media",
        ]].to_string(index=False))
    else:
        pref_bin = pd.DataFrame()

    # Matrizes de confusão para RF original e Park na melhor janela geral.
    for metodo in ["RF_original_wide", "Park"]:
        plot_confusion_from_predictions(df_pred_multi, best_multi_j, metodo, FEATURE_SET_PREFERIDO, binario=False)
        if len(df_pred_bin) > 0:
            plot_confusion_from_predictions(df_pred_bin, best_multi_j, metodo, FEATURE_SET_PREFERIDO, binario=True)

    # Métricas por temperatura e curvas exemplo.
    if len(df_metricas) > 0:
        plot_metricas_por_temperatura(df_metricas, best_multi_j, ["RF_original_wide", "Park"])

    row_window_plot = df_windows[df_windows["janela_id"] == best_multi_j].iloc[0]
    plot_curvas_exemplo(df_base, row_window_plot, TEMP_EXEMPLO_CURVAS)

    print("\n" + "=" * 100)
    print("✅ TESTE FINALIZADO")
    print(f"📁 Pasta de saída: {PASTA_SAIDA}")
    print(f"⏱️ Tempo total: {time.time() - t0_total:.1f} s")
    print("=" * 100)

    return {
        "df_windows": df_windows,
        "df_multi": df_multi,
        "df_bin": df_bin,
        "df_metricas": df_metricas,
        "df_pred_multi": df_pred_multi,
        "df_pred_bin": df_pred_bin,
        "df_resumo_multi": df_resumo_multi,
        "df_resumo_bin": df_resumo_bin,
    }


# ============================================================
# 12) RODAR
# ============================================================

if __name__ == "__main__":
    resultados = executar_teste()
